
## 🔄 Day 1 복습 퀴즈 — 핵심 다시 만들기

어제 만든 `scaled_dot_product_attention` 함수를 **기억을 더듬어** 다시 완성해 보세요. <br>
(Linear projection 정의 → input 을 Linear Projection 통해 Q,K,V 로 전환 → Q와 K의 유사도 → ② sqrt(d_k)로 나누기 → ③ softmax → ④ V와 가중합) <br>
(입력 텐서도 만들어 보세요)


In [ ]:
import math
import torch
import torch.nn as nn

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용할 device:", device)

embedding_dim = 64

# Q, K, V 에 대한 Linear Projection 정의
query_projection = 
key_projection = 
value_projection = 
output_projection = 

# 어제 만든 함수를 다시 완성해 보세요! (4단계를 떠올리며)
def scaled_dot_product_attention(input_tensor):
    """값을 채워보세요"""
    # Linear Projection
    query = 
    key = 
    value = 

    # Attention score (유사도)
    attention_scores = 
    
    # Scaling
    d_k = key.shape[-1]
    scaled_scores = 

    # Attention_weight (가중치화 (확률로 바꾸기))
    attention_weights = 

    # Attention 결과
    attention_output = 

    # Output Linear Projection
    attention_output = 

    return attention_output, attention_weights

# 랜덤값의 64 차원(채널) 토큰 6개짜리 tensor 생성
input_tensor = 

print("연습용 input tensor shape:", input_tensor.shape)

output, weights = scaled_dot_product_attention(input_tensor)
print("output shape:", output.shape, "attention weight shape:", weights.shape)

---
# Part 1. Multi-Head Attention — 여러 관점에서 동시에 보기

어제 만든 어텐션은 문장을 **하나의 관점**으로만 봅니다.
하지만 "cat"이라는 단어는 문법 관점(주어인가?), 의미 관점(동물인가?), 관계 관점(누구의 고양이인가?)
등 **여러 관점**에서 볼 수 있어야 합니다. 실제 트랜스포머는 어텐션을 head 여러 개로 나눠 동시에 돌립니다.

1. 64차원 임베딩을 **16차원씩 4조각**으로 나눈다 (head마다 한 조각씩 담당)
2. 각 조각에 **따로따로** 어제의 어텐션을 돌린다
3. 결과 4조각을 다시 **이어 붙인다(concat)** → 다시 64차원

## 1.1 head 나누기 — 일단 눈으로 확인

In [ ]:
# input_tensor (6, 64)를 마지막 차원(dim=-1) 기준 4조각으로
head_1, head_2, head_3, head_4 = 

print("head_1 shape:", head_1.shape)
print("head_2 shape:", head_2.shape)
print("head_3 shape:", head_3.shape)
print("head_4 shape:", head_4.shape)

## 1.2 각 head별로 어텐션 돌리기 → 다시 합치기

### 4조각 각각에 `scaled_dot_product_attention`을 적용하고, `torch.cat`으로 이어 붙이기


In [ ]:
# 각 head에 self-attention 적용 (Q=K=V=그 head 조각)
head_1_output, head_1_weights = 
head_2_output, head_2_weights = 
head_3_output, head_3_weights = 
head_4_output, head_4_weights = 

print("head별 출력 shape:", head_1_output.shape, head_2_output.shape, head_3_output.shape, head_4_output.shape)

# 4개 출력을 마지막 차원으로 이어 붙이기 → 원래 크기 (6, 64) 복원
multi_head_output = 

print("합친 결과 shape:", multi_head_output.shape)   # 예상: (6, 64)
# 각 head는 자기 조각 안에서 "독립적으로" 어텐션을 계산했습니다 — 이게 멀티헤드의 핵심

## 1.3 reshape + transpose

### head가 8개라면? 16개라면? → **"한 번에 병렬처리"**

In [ ]:
print("원본 shape:", input_tensor.shape)
print("T shape:", )
print("Transpose shape:", )

print("Reshape shape:", )
print("Reshape + Transpose shape:", )

In [ ]:
# head 를 torch.chunk 를 통해 나누고 stack 한 결과

print(stacked_tensor.shape)

# reshape + transpose 활용 결과
reshaped_transposed_tensor = 
print(reshaped_transposed_tensor.shape)

torch.equal(stacked_tensor, reshaped_transposed_tensor)

## 1.4 Reshape 과 Transpose 를 활용하여 Multi-head Attention 구현하기

head 는 4개 로 진행

In [ ]:
embedding_dim = 64

query_projection = 
key_projection = 
value_projection = 
output_projection = 

def multi_head_attention(input_tensor):
    """값을 채워보세요"""
    num_heads = 
    head_dim = 
    sequence_length = input_tensor.shape[0]

    # Apply Linear Projection
    query = 
    key = 
    value = 

    # multi-head split (hint: reshape + transpose)


    # Attention score (유사도 계산) 
    # hint: query 는 [head, token, channel], key 는 [head, channel, token] 이 되어야함
    attention_scores = 
    
    # Scaling
    d_k = key.shape[-1]
    scaled_scores = 

    # Attention_weight (가중치화 (확률로 바꾸기))
    attention_weights = 

    # Attention 결과
    attention_output = 

    # 중간 shape 확인
    print("mult-head attention shape", attention_output.shape)

    # 원래 상태로 되돌리기 (multi-head → single)
    attention_output = 

    # Output Projection
    attention_output = 

    return attention_output, attention_weights

attention_output, attention_weights = multi_head_attention(input_tensor)

print("multi-head attention weight output:", attention_weights.shape, "← torch.Size([head, token, token])")
print("multi-head attention output:", attention_output.shape, "← torch.Size([head, token, channel])")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, ax in enumerate(axes):
    image = ax.imshow(
        attention_weights[i].detach(),
        cmap="viridis",
    )

    ax.set_title(f"Head {i + 1}")
    ax.set_xlabel("Key token")
    ax.set_ylabel("Query token")

    ax.set_xticks(range(6))
    ax.set_yticks(range(6))

# 네 그림이 공유하는 colorbar
fig.colorbar(image, shrink=0.8, label="Attention weight")

plt.tight_layout()
plt.show()


## Example (학습된 모델의 실제 qk similarity 를 multi-head 관점에서 확인)

<div style="display: flex; gap: 12px;">
  <img src="dummy_tensor/pisa.png" width="500">
  <img src="dummy_tensor/car.jpg" width="500">
</div>

In [ ]:
q = torch.load("dummy_tensor/pisa_q.pt")
k = torch.load("dummy_tensor/pisa_k.pt")

print(q.shape)

q = q.flatten(0,1)
k = k.flatten(0,1)
multi_head_sim = q.reshape(3750, 6, 64).transpose(0, 1) @ k.reshape(3750, 6, 64).permute(1, 2, 0)

In [ ]:
import matplotlib.pyplot as plt
# 차 2820
# 우측 건물 1490
# 좌측 건물 1000

# 피사 1025
# 조형물 3030
# 잔디: 3500

token_idx = 3500
height, width = 50, 75

# sim shape: [6, 3750, 3750]
sim_cpu = multi_head_sim.detach().cpu()

row = token_idx // width
col = token_idx % width

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

# 모든 head에서 색상 범위를 동일하게 설정
selected_maps = sim_cpu[:, token_idx].reshape(6, height, width)
vmin = selected_maps.min()
vmax = selected_maps.max()

for head_idx, ax in enumerate(axes):
    similarity_map = selected_maps[head_idx]

    image = ax.imshow(
        similarity_map,
    )

    # 현재 query token 위치 표시
    ax.scatter(
        col,
        row,
        s=100,
        c="red",
        marker="o",
        edgecolors="white",
        linewidths=1.5
    )

    ax.set_title(f"Head {head_idx + 1}")
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(
    f"Attention maps for token {token_idx}",
    fontsize=16
)

plt.tight_layout()
plt.show()